# 01 · Country-Level Monthly Stats Exploration

One row per country per month — the broadest geographic granularity and the best
starting point for global comparisons.

**What you will learn:**
- How download, upload, latency, and loss vary across countries
- How to read the full percentile distribution (not just the median)
- What the "percentile polarity inversion" means for latency and loss
- Why download speed and upload speed are often highly asymmetric

## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

try:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import seaborn as sns
    from IPython.display import clear_output, display
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 5)
except ImportError as e:
    print(f"Note: {e}")
    print("  Install with: uv add matplotlib seaborn ipywidgets")


In [2]:
# ── Country name lookup ──────────────────────────────────────────────────────
# countrylookup.py is a local helper (same directory as this notebook) that
# converts ISO 3166-1 alpha-2 codes to readable English country names.
# It tries pycountry → restcountries.com API → built-in fallback dict.
#
# If you move this notebook, keep countrylookup.py alongside it.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # ensure local module is found
from countrylookup import cc_name, cc_label

print(f"Country lookup ready — {cc_label('US')}, {cc_label('KR')}, {cc_label('XK')}")

[countrylookup] downloading country names from restcountries.com ...
Country lookup ready — United States (US), South Korea (KR), Kosovo (XK)


In [3]:
# ── Discover available months ─────────────────────────────────────────────────
#
# Fetch the M-Lab manifest to learn which months are available per slice.
# Each entry includes the public download URL and the local cache path.

MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()

records = []
for path, meta in resp.json()["files"].items():
    parts = path.split("/")
    if len(parts) >= 6 and parts[5] == "data.parquet":
        records.append({
            "start":      pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "slice":      parts[4],
            "url":        meta["url"],
            "cache_path": path,
        })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)

print(f"Catalog: {len(catalog)} entries, {catalog['slice'].nunique()} slices, "
      f"{catalog['start'].min().date()} → {catalog['start'].max().date()}")


Catalog: 2450 entries, 12 slices, 2009-01-01 → 2026-01-01


# Data Loader 

The below function is used to downoad and open the parquet files based on the start date of the file, then cache it locally so we aren't re-downloading data again and again. It checks for data in the cache first before it downloads the data again. 

In [4]:
# ── Data loader ──────────────────────────────────────────────────────────────
#
# Downloads parquet files from the public M-Lab URLs in the manifest.
# Files are cached to ./cache/v1/... on first access and reused on subsequent
# runs (matching the path structure used by the iqb library's local cache).

from io import BytesIO

_mem_cache: dict = {}

def load_parquet(slice_name: str, start: str) -> pd.DataFrame:
    key = (slice_name, start)
    if key in _mem_cache:
        return _mem_cache[key]

    start_ts = pd.to_datetime(start)
    row = catalog[(catalog["slice"] == slice_name) & (catalog["start"] == start_ts)]
    if row.empty:
        available = catalog[catalog["slice"] == slice_name]["start"].dt.strftime("%Y-%m-%d").tolist()
        raise ValueError(f"No data for slice='{slice_name}', month='{start}'.\nAvailable: {available}")
    row = row.iloc[0]

    local_path = Path(row["cache_path"])
    if local_path.exists():
        df = pd.read_parquet(local_path)
    else:
        print(f"[download] {slice_name} / {start} …")
        r = requests.get(row["url"], timeout=60)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        df = pd.read_parquet(BytesIO(r.content))
        print(f"  ✓ saved to {local_path}  ({len(df):,} rows)")

    _mem_cache[key] = df
    return df


## Look at a sample of the data

We can load the data from September of 2025 individually using the first day of the month `2025-09-01` as below. Then look at the first 5 rows to get a sense of how the data tables in the parquet files are laid out.

In [5]:
explore = load_parquet("downloads_by_country",'2025-09-01' )
explore.columns
explore.head(5)

,country_code,sample_count,download_p1,download_p5,download_p10,download_p25,download_p50,download_p75,download_p90,download_p95,...,latency_p99,loss_p1,loss_p5,loss_p10,loss_p25,loss_p50,loss_p75,loss_p90,loss_p95,loss_p99
0,AD,1484,0.725282,8.644480,23.395465,49.593681,115.467154,298.217298,402.513711,601.143099,...,0.212,0.160657,0.113175,0.099363,0.025152,0.000073,0.000000,0.0,0.0,0.0
1,AE,201121,0.115003,0.399648,1.117642,6.891370,30.227920,94.319410,250.431997,395.171552,...,14.456,0.386054,0.293968,0.233096,0.126742,0.024452,0.000027,0.0,0.0,0.0
2,AF,5308,0.053123,0.123862,0.212373,0.559634,1.515375,4.130586,10.998606,24.108041,...,49.000,0.398626,0.335549,0.297792,0.222200,0.138865,0.019703,0.0,0.0,0.0
3,AG,630,0.833483,3.749902,6.994399,20.917504,46.070122,68.260270,113.137162,121.519742,...,83.334,0.352666,0.253725,0.187098,0.091748,0.011876,0.000073,0.0,0.0,0.0
4,AI,150,0.481449,2.689346,7.294242,13.741709,51.588683,132.831244,203.219632,212.722703,...,32.929,0.406171,0.194201,0.132457,0.059193,0.001935,0.000000,0.0,0.0,0.0


## Interactive Country Explorer

Now that we understand the data structure a bit, lets build an interactive country explorer with the data. 

Choose a month and metric, then adjust how many countries appear.

> **Tip:** `_p50` is the median — the midpoint of all tests that month, robust to
> a small number of very fast or very slow connections. It is the most meaningful
> single-number summary in this dataset.

In [6]:
country_months = sorted(
    catalog[catalog["slice"] == "downloads_by_country"]["start"]
    .dt.strftime("%Y-%m-%d").unique(), reverse=True,
)

# Metric options: display label → (column name, lower_is_better)
# Latency and loss: lower raw values are better, but IQB's polarity inversion
# means higher percentiles represent the better-performing connections.
METRICS = {
    "Download p50 (Mbit/s)":       ("download_p50",  False),
    "Upload p50 (Mbit/s)":         ("upload_p50",    False),
    "Latency p50 (ms)":            ("latency_p50",   True),
    "Packet Loss p50 (fraction)":  ("loss_p50",      True),
}

w_month  = widgets.Dropdown(options=country_months, description="Month:",
                             layout=widgets.Layout(width="250px"))
w_metric = widgets.Dropdown(options=list(METRICS),  description="Metric:",
                             layout=widgets.Layout(width="290px"))
w_topn   = widgets.IntSlider(value=20, min=5, max=60, step=5, description="Top N:",
                              layout=widgets.Layout(width="380px"))
out      = widgets.Output()

def update(change=None):
    month = w_month.value
    col, lower = METRICS[w_metric.value]
    n = w_topn.value
    dl = load_parquet("downloads_by_country", month)
    ul = load_parquet("uploads_by_country",   month)
    df = dl.merge(ul[["country_code","upload_p50"]], on="country_code", how="left")
    top = df.nsmallest(n, col) if lower else df.nlargest(n, col)
    top = top.sort_values(col, ascending=not lower)
    top = top.copy()
    top["country_label"] = top["country_code"].apply(cc_name)
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, max(5, n * 0.36)))
        ax.barh(top["country_label"], top[col])
        note = "(lower = better)" if lower else ""
        ax.set_xlabel(w_metric.value)
        ax.set_title(f"Top {n} countries — {w_metric.value} {note}\n{month}")
        plt.tight_layout(); plt.show()

w_month.observe(update, "value"); w_metric.observe(update, "value")
w_topn.observe(update, "value")
display(widgets.VBox([widgets.HBox([w_month, w_metric]), w_topn, out]))
update()

## Percentile Distribution

Monthly Stats stores nine percentile points (p1–p99) per metric. A wide spread between low
and high percentiles signals *inequality*: some users have very fast connections
while others in the same country have very slow ones.

> **Polarity reminder** — For latency and loss, higher percentiles represent
> *better* (lower-latency, lower-loss) connections due to the dataset's inversion.

In [7]:
PERCENTILE_METRICS = [
    ("Download (Mbit/s)", "download"),
    ("Upload (Mbit/s)",   "upload"),
    ("Latency (ms)",      "latency"),
    ("Packet Loss",       "loss"),
]

w_pm  = widgets.Dropdown(options=country_months, description="Month:",
                          layout=widgets.Layout(width="250px"))
w_pme = widgets.Dropdown(options=[l for l,_ in PERCENTILE_METRICS], description="Metric:",
                          layout=widgets.Layout(width="260px"))

_pct_data: dict = {}
def _get_df(month):
    if month not in _pct_data:
        _pct_data[month] = load_parquet("downloads_by_country", month)
    return _pct_data[month]

w_pcountries = widgets.SelectMultiple(
    options=sorted(_get_df(country_months[0])["country_code"].dropna().unique()),
    value=["US","DE","BR","IN","JP"], description="Countries:", rows=8,
    layout=widgets.Layout(width="200px"),
)
out_pct = widgets.Output()

def update_pct(change=None):
    month  = w_pm.value
    prefix = dict(PERCENTILE_METRICS)[w_pme.value]
    df     = _get_df(month)
    pcols  = sorted([c for c in df.columns if c.startswith(f"{prefix}_p")],
                    key=lambda c: int(c.split("_p")[1]))
    pnums  = [int(c.split("_p")[1]) for c in pcols]
    with out_pct:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        for cc in w_pcountries.value:
            row = df[df["country_code"] == cc]
            if row.empty: continue
            ax.plot(pnums, [float(row.iloc[0][c]) for c in pcols], marker="o", label=cc_label(cc))
        note = ("\n(IQB polarity inverted: p95 = best connections, p5 = worst)"
                if prefix in ("latency","loss") else "")
        ax.set_xlabel("Percentile"); ax.set_ylabel(w_pme.value)
        ax.set_title(f"Percentile distribution — {w_pme.value} — {month}{note}")
        ax.legend(title="Country"); plt.tight_layout(); plt.show()

def on_pm_change(change):
    prev = list(w_pcountries.value)  # remember current multi-selection
    new_opts = sorted(_get_df(change["new"])["country_code"].dropna().unique())
    w_pcountries.options = new_opts
    # Restore any previously selected countries that still exist
    restored = [c for c in prev if c in new_opts]
    w_pcountries.value = restored if restored else [new_opts[0]]
    update_pct()

w_pm.observe(on_pm_change,"value"); w_pme.observe(update_pct,"value")
w_pcountries.observe(update_pct,"value")
display(widgets.VBox([widgets.HBox([w_pm, w_pme]),
                      widgets.HBox([w_pcountries, out_pct])]))
update_pct()

## Download vs Upload Scatter

Most residential broadband is *asymmetric*: much faster download than upload.
Points above the dashed diagonal have faster upload than download — often fibre
or business-grade connections. Colour encodes median latency.

In [8]:
w_sm = widgets.Dropdown(options=country_months, description="Month:",
                         layout=widgets.Layout(width="250px"))
out_s = widgets.Output()

def update_scatter(change=None):
    month = w_sm.value
    dl = load_parquet("downloads_by_country", month)
    ul = load_parquet("uploads_by_country",   month)
    df = dl.merge(ul[["country_code","upload_p50"]], on="country_code", how="inner")
    with out_s:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8,8))
        sc = ax.scatter(df["download_p50"], df["upload_p50"],
                        c=df["latency_p50"], cmap="RdYlGn_r", alpha=0.65,
                        edgecolors="white", linewidths=0.5)
        lim = max(df["download_p50"].max(), df["upload_p50"].max()) * 1.05
        ax.plot([0,lim],[0,lim],"k--",lw=0.8,alpha=0.4,label="upload = download")
        plt.colorbar(sc, ax=ax, label="Median latency (ms)")
        ax.set_xlabel("Median download (Mbit/s)"); ax.set_ylabel("Median upload (Mbit/s)")
        ax.set_title(f"Download vs upload — all countries — {month}\n"
                     "Above dashed line = upload faster than download")
        ax.legend(); plt.tight_layout(); plt.show()

w_sm.observe(update_scatter,"value")
display(widgets.VBox([w_sm, out_s]))
update_scatter()